# SmartFit AI: Diet Plan Generator (Customized for FoodData Central)

### **Goal:** To create a rule-based engine that generates a personalized daily meal plan using the `fda_approved_food_items_w_nutrient_info.csv` dataset.

**This notebook will perform the following steps:**
1.  **Load and Clean:** Load the dataset and preprocess it. This involves selecting relevant columns, renaming them for clarity, and handling missing values.
2.  **Define Nutritional Logic:** Implement standard formulas for calculating a user's Total Daily Energy Expenditure (TDEE) and macronutrient targets.
3.  **Build the Generator:** Create a function that takes a user's nutritional targets and uses a greedy algorithm to build a meal plan. The logic will be specifically designed to work with nutrient values provided **per 100g**.
4.  **Test the Generator:** Run the entire pipeline with a sample user profile to generate a full day's meal plan and verify the results.

In [82]:
import pandas as pd

# Full path to your CSV file on your local machine
file_path = r'C:\Users\MUHAMMAD ZAIN\Desktop\SMART_FIT\data_sets\fda_approved_food_items_w_nutrient_info.csv'

try:
    df_food = pd.read_csv(file_path)
    print("Dataset loaded successfully!")
except FileNotFoundError:
    print(f"Error: File not found at '{file_path}'")
    df_food = None


C:\Users\MUHAMMAD ZAIN\AppData\Local\Temp\ipykernel_14996\1155416511.py:7: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_food = pd.read_csv(file_path)


Dataset loaded successfully!


In [83]:
df_food.head()

,fdc_id,brand_owner,description,ingredients,gtin_upc,serving_size,serving_size_unit,branded_food_category,modified_date,available_date,...,Thiamin-MG,Total lipid (fat)-G,Total sugar alcohols-G,"Vitamin A, IU-IU",Vitamin B-12-UG,Vitamin B-6-MG,"Vitamin C, total ascorbic acid-MG","Vitamin D (D2 + D3), International Units-IU",Vitamin D (D2 + D3)-UG,"Zinc, Zn-MG"
0,1105904,Richardson Oilseed Products (US) Limited,WESSON Vegetable Oil 1 GAL,Vegetable Oil,27000612323,15.0,ml,Oils Edible,2020-10-02,2020-11-13,...,NaN,93.33,NaN,0.0,NaN,NaN,0.0,0.0,NaN,NaN
1,1105905,CAMPBELL SOUP COMPANY,SWANSON BROTH BEEF,"INGREDIENTS: BEEF STOCK, CONTAINS LESS THAN 2%...",51000198808,240.0,ml,Herbs/Spices/Extracts,2020-09-12,2020-11-13,...,NaN,0.00,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
2,1105906,CAMPBELL SOUP COMPANY,CAMPBELL'S SLOW KETTLE SOUP CLAM CHOWDER,"INGREDIENTS: CLAM STOCK, POTATOES, CLAMS, CREA...",51000213273,440.0,g,Prepared Soups,2020-09-01,2020-11-13,...,NaN,5.31,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
3,1105907,CAMPBELL SOUP COMPANY,CAMPBELL'S SLOW KETTLE SOUP CHEESE BROCCOLI,"INGREDIENTS: WATER, CREAM, BROCCOLI, CELERY, V...",51000213303,440.0,g,Prepared Soups,2020-09-01,2020-11-13,...,NaN,6.12,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
4,1105908,CAMPBELL SOUP COMPANY,SWANSON BROTH CHICKEN,"INGREDIENTS: CHICKEN STOCK, CONTAINS LESS THAN...",51000224637,240.0,ml,Herbs/Spices/Extracts,2020-10-03,2020-11-13,...,NaN,0.00,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN


In [84]:
import pandas as pd
import random
import json
import numpy as np

### Step 2: Data Exploration and Preprocessing

The FoodData Central dataset is very rich but also complex. We need to simplify it for our goal.

**Our plan:**
1.  **Select essential columns:** We only need the food's description and its core nutrient values (Calories, Protein, Fat, Carbs).
2.  **Rename columns:** Use simpler names for easier access (e.g., `Energy_kcal` -> `Calories`).
3.  **Handle data types:** Ensure all nutrient columns are numeric.
4.  **Filter invalid data:** Remove rows where essential nutrients are missing or are zero, as these are not useful for meal planning.

In [86]:
# @title Step 2: Data Cleaning (FIXED & MORE ROBUST)
if df_food is not None:
    # --- DIAGNOSTIC STEP: Print all available columns to find the correct names ---
    print("--- All Columns in your CSV file ---")
    print(df_food.columns.tolist())
    print("-" * 35)

    # 1. Define the columns we want to keep and their new names.
    #    I have updated the keys based on the likely correct names.
    #    The original names are on the left, our new, simple names are on the right.
    columns_to_keep = {
        'description': 'Food',
        'branded_food_category': 'Category',
        'Energy-KCAL': 'Calories',         #<-- LIKELY FIX 1: Changed 'Energy_kcal' to 'Energy-KCAL'
        'Protein-G': 'Protein',           #<-- LIKELY FIX 2: Changed 'Protein_g' to 'Protein-G'
        'Total lipid (fat)-G': 'Fat',
        'Carbohydrate, by difference-G': 'Carbs'
    }

    # Check if all our expected old names exist in the dataframe's columns
    missing_cols = [col for col in columns_to_keep.keys() if col not in df_food.columns]
    if missing_cols:
        print(f"\nERROR: The following columns were not found in your file: {missing_cols}")
        print("Please check the column list printed above and update the 'columns_to_keep' dictionary in this cell.")
    else:
        # 2. Select and rename the columns
        df_food_cleaned = df_food[list(columns_to_keep.keys())].rename(columns=columns_to_keep)

        # 3. Convert nutrient columns to numeric, forcing errors to NaN
        nutrient_cols = ['Calories', 'Protein', 'Fat', 'Carbs']
        for col in nutrient_cols:
            df_food_cleaned[col] = pd.to_numeric(df_food_cleaned[col], errors='coerce')

        # 4. Drop rows where any of the essential nutrients are missing (NaN)
        df_food_cleaned.dropna(subset=nutrient_cols, inplace=True)

        # 5. Remove foods with 0 calories
        df_food_cleaned = df_food_cleaned[df_food_cleaned['Calories'] > 0].copy()

        # 6. Reset index after dropping rows
        df_food_cleaned.reset_index(drop=True, inplace=True)

        print("\n--- Cleaned and Processed Food DataFrame ---")
        print(f"Dataset reduced to {len(df_food_cleaned)} usable food items.")
        df_food_cleaned.info()
        print("\nSample of cleaned data:")
        print(df_food_cleaned.head())

--- All Columns in your CSV file ---
['fdc_id', 'brand_owner', 'description', 'ingredients', 'gtin_upc', 'serving_size', 'serving_size_unit', 'branded_food_category', 'modified_date', 'available_date', 'Calcium, Ca-MG', 'Carbohydrate, by difference-G', 'Cholesterol-MG', 'Copper, Cu-MG', 'Energy-KCAL', 'Fatty acids, total monounsaturated-G', 'Fatty acids, total polyunsaturated-G', 'Fatty acids, total saturated-G', 'Fatty acids, total trans-G', 'Fiber, soluble-G', 'Fiber, total dietary-G', 'Folate, total-UG', 'Folic acid-UG', 'Iron, Fe-MG', 'Magnesium, Mg-MG', 'Manganese, Mn-MG', 'Niacin-MG', 'Pantothenic acid-MG', 'Phosphorus, P-MG', 'Potassium, K-MG', 'Protein-G', 'Riboflavin-MG', 'Sodium, Na-MG', 'Sugars, added-G', 'Sugars, total including NLEA-G', 'Thiamin-MG', 'Total lipid (fat)-G', 'Total sugar alcohols-G', 'Vitamin A, IU-IU', 'Vitamin B-12-UG', 'Vitamin B-6-MG', 'Vitamin C, total ascorbic acid-MG', 'Vitamin D (D2 + D3), International Units-IU', 'Vitamin D (D2 + D3)-UG', 'Zinc, Zn-

### Step 3: Define Nutritional Logic & Formulas

This section defines the scientific foundation for our diet planner. We'll create two key functions:

1.  **`calculate_tdee`**: This function will implement the well-known Harris-Benedict Equation to estimate a user's Total Daily Energy Expenditure (TDEE), which is the number of calories they burn per day.
2.  **`calculate_macros`**: Based on the TDEE and the user's specific goal (e.g., fat loss, muscle gain), this function will calculate the target daily intake of macronutrients: protein, carbohydrates, and fat.

In [87]:
# @title TDEE and Macro Calculation Functions
def calculate_tdee(gender, weight_kg, height_cm, age, activity_level):
    """
    Calculates Total Daily Energy Expenditure (TDEE) using Harris-Benedict Equation.
    Activity Levels (multiplier):
    - 1.2: Sedentary (little or no exercise)
    - 1.375: Lightly active (light exercise/sports 1-3 days/week)
    - 1.55: Moderately active (moderate exercise/sports 3-5 days/week)
    - 1.725: Very active (hard exercise/sports 6-7 days a week)
    """
    if gender.lower() == 'male':
        bmr = 88.362 + (13.397 * weight_kg) + (4.799 * height_cm) - (5.677 * age)
    else: # assumes female
        bmr = 447.593 + (9.247 * weight_kg) + (3.098 * height_cm) - (4.330 * age)

    return bmr * activity_level

def calculate_macros(tdee, goal):
    """
    Calculates macronutrient targets based on TDEE and a goal.
    Goals: 'fat_loss' (-500 kcal), 'muscle_gain' (+500 kcal), 'maintenance'
    """
    if goal == 'fat_loss':
        calorie_target = tdee - 500
    elif goal == 'muscle_gain':
        calorie_target = tdee + 500
    else: # maintenance
        calorie_target = tdee

    # Standard macro split: 40% Carbs, 30% Protein, 30% Fat
    # 1g Protein = 4 kcal, 1g Carb = 4 kcal, 1g Fat = 9 kcal
    macros = {
        'calories': calorie_target,
        'protein': (calorie_target * 0.30) / 4,
        'carbs': (calorie_target * 0.40) / 4,
        'fat': (calorie_target * 0.30) / 9
    }
    # Return rounded values
    return {k: round(v) for k, v in macros.items()}

# Quick test to ensure the functions work
test_tdee = calculate_tdee('male', 80, 180, 30, 1.55)
test_macros = calculate_macros(test_tdee, 'muscle_gain')
print(f"Test TDEE: {round(test_tdee)}")
print(f"Test Macros: {test_macros}")

Test TDEE: 2873
Test Macros: {'calories': 3373, 'protein': 253, 'carbs': 337, 'fat': 112}


### Step 4: Build the Meal Generator Function

This is the core "AI" of our diet planner. We will create the `generate_diet_plan` function that uses a **greedy algorithm** to assemble a meal plan.

**How it works:**
- It takes the user's daily nutrient targets as input.
- It iterates through each meal (Breakfast, Lunch, Dinner), allocating a percentage of the daily calories to each.
- For each meal, it intelligently selects several food items from our cleaned dataset. It calculates a realistic portion size (in grams) for each food to try and meet the meal's calorie target.
- The function is designed to be "smart" by calculating portion sizes based on the food's nutrient density, rather than just picking a random amount.

In [88]:
# @title The Diet Plan Generator Function
import random
import json

def generate_diet_plan(df, daily_targets):
    """
    Generates a daily meal plan using a greedy algorithm.
    All nutrient values in the dataframe are per 100g.

    Args:
        df (pd.DataFrame): The cleaned food dataframe.
        daily_targets (dict): A dictionary with target 'calories', 'protein', 'carbs', 'fat'.

    Returns:
        dict: A structured meal plan.
    """
    meal_plan = {
        'Breakfast': {'items': [], 'totals': {'Calories': 0, 'Protein': 0, 'Fat': 0, 'Carbs': 0}},
        'Lunch': {'items': [], 'totals': {'Calories': 0, 'Protein': 0, 'Fat': 0, 'Carbs': 0}},
        'Dinner': {'items': [], 'totals': {'Calories': 0, 'Protein': 0, 'Fat': 0, 'Carbs': 0}},
    }
    # Meal calorie distribution
    meal_distribution = {'Breakfast': 0.25, 'Lunch': 0.40, 'Dinner': 0.35}

    for meal_name, percentage in meal_distribution.items():
        meal_targets_calories = daily_targets['calories'] * percentage
        meal_df = meal_plan[meal_name]

        # Add 2-3 items per meal
        for _ in range(random.randint(2, 3)):
            # Calculate remaining calories needed for this meal
            calories_needed = meal_targets_calories - meal_df['totals']['Calories']
            if calories_needed <= 50: # If meal is already full, stop
                break

            # Find a suitable food item
            # We want an item that can fill a decent portion of the remaining calories
            # Let's target a food that has between 100 and 500 calories per 100g to get variety
            potential_foods = df[
                (df['Calories'] > 100) & (df['Calories'] < 500)
            ]
            if potential_foods.empty: # Fallback if no foods in range
                potential_foods = df[df['Calories'] > 0]

            food_sample = potential_foods.sample(1).iloc[0]

            # Calculate portion size to meet a fraction of the remaining calories
            # This heuristic prevents one food item from dominating the meal
            num_items_left_to_add = (3 - len(meal_df['items']))
            target_cals_for_item = calories_needed / num_items_left_to_add if num_items_left_to_add > 0 else calories_needed

            # Portion size (g) = (Target Calories / Calories per 100g) * 100
            portion_g = round((target_cals_for_item / food_sample['Calories']) * 100)
            # Clamp portion size to a realistic range (e.g., 50g to 350g)
            portion_g = min(max(portion_g, 50), 350)

            portion_multiplier = portion_g / 100.0

            item = {
                'Food': food_sample['Food'],
                'Portion (g)': portion_g,
                'Calories': round(food_sample['Calories'] * portion_multiplier),
                'Protein': round(food_sample['Protein'] * portion_multiplier),
                'Fat': round(food_sample['Fat'] * portion_multiplier),
                'Carbs': round(food_sample['Carbs'] * portion_multiplier)
            }

            meal_df['items'].append(item)
            # Update meal totals
            for key in ['Calories', 'Protein', 'Fat', 'Carbs']:
                meal_df['totals'][key] += item[key]

    return meal_plan

### Step 5: Test the Full Pipeline

This is the final step where we bring everything together. We will simulate a real user request from start to finish.

**The process:**
1.  **Define a sample user profile** (you can edit this to see different results).
2.  Call our functions to **calculate their unique nutritional needs**.
3.  Pass these needs to the **`generate_diet_plan` function**.
4.  **Print the final, structured meal plan** and a summary comparing the plan's totals to the user's targets.

In [89]:
# @title Example Usage: Generating a Plan for a Sample User
if 'df_food_cleaned' in locals() and not df_food_cleaned.empty:
    # --- 1. Define Sample User Profile ---
    # You can change these values to test different scenarios!
    user_profile = {
        'gender': 'female',
        'weight_kg': 65,
        'height_cm': 170,
        'age': 25,
        'activity_level': 1.375, # Lightly active
        'goal': 'fat_loss'       # Options: 'muscle_gain', 'fat_loss', 'maintenance'
    }

    # --- 2. Calculate TDEE and Macros ---
    print("--- Calculating User's Nutritional Needs ---")
    user_tdee = calculate_tdee(
        user_profile['gender'],
        user_profile['weight_kg'],
        user_profile['height_cm'],
        user_profile['age'],
        user_profile['activity_level']
    )
    print(f"User's estimated TDEE for maintenance: {round(user_tdee)} kcal")

    user_macros = calculate_macros(user_tdee, user_profile['goal'])
    print(f"\nUser's Daily Targets for '{user_profile['goal']}':")
    print(json.dumps(user_macros, indent=2))

    # --- 3. Generate the Diet Plan ---
    print("\n--- Generating Diet Plan... ---")
    final_diet_plan = generate_diet_plan(df_food_cleaned, user_macros)

    print("\n\n" + "="*20 + " GENERATED DIET PLAN " + "="*20)
    print(json.dumps(final_diet_plan, indent=2))

    # --- 4. Review and Compare Totals ---
    total_calories = sum(final_diet_plan[m]['totals']['Calories'] for m in final_diet_plan)
    total_protein = sum(final_diet_plan[m]['totals']['Protein'] for m in final_diet_plan)
    total_carbs = sum(final_diet_plan[m]['totals']['Carbs'] for m in final_diet_plan)
    total_fat = sum(final_diet_plan[m]['totals']['Fat'] for m in final_diet_plan)

    print("\n\n" + "="*20 + " PLAN TOTALS vs. TARGETS " + "="*20)
    print(f"Calories: {total_calories} / {user_macros['calories']} kcal")
    print(f"Protein:  {total_protein}g / {user_macros['protein']}g")
    print(f"Carbs:    {total_carbs}g / {user_macros['carbs']}g")
    print(f"Fat:      {total_fat}g / {user_macros['fat']}g")

else:
    print("Error: `df_food_cleaned` DataFrame not found or is empty. Please run the cleaning steps first.")

--- Calculating User's Nutritional Needs ---
User's estimated TDEE for maintenance: 2017 kcal

User's Daily Targets for 'fat_loss':
{
  "calories": 1517,
  "protein": 114,
  "carbs": 152,
  "fat": 51
}

--- Generating Diet Plan... ---


==================== GENERATED DIET PLAN ====================
{
  "Breakfast": {
    "items": [
      {
        "Food": "BUTTER CHICKEN MINI NAANPANADAS A MINI EMPANADA FILLED WITH BUTTER CHICKEN & CHEESE, BUTTER CHICKEN",
        "Portion (g)": 51,
        "Calories": 128,
        "Protein": 6,
        "Fat": 4,
        "Carbs": 18
      },
      {
        "Food": "CRISPY MARSHMALLOW SQUARES",
        "Portion (g)": 50,
        "Calories": 204,
        "Protein": 2,
        "Fat": 6,
        "Carbs": 39
      }
    ],
    "totals": {
      "Calories": 332,
      "Protein": 8,
      "Fat": 10,
      "Carbs": 57
    }
  },
  "Lunch": {
    "items": [
      {
        "Food": "CHOCOLATE CHIP MUFFINS",
        "Portion (g)": 54,
        "Calories": 202,
   

### Step 6: Improving the Algorithm (Protein-Focused)

**Analysis:** Our first-generation algorithm successfully created a plan that was close in calories, but it significantly missed the protein target. This is because it was only optimized to fill a calorie budget.

**Improvement:** We will now create a `v2` of our generator. This version will use a **protein-focused greedy strategy**:
1.  For each meal, it will first calculate the protein target.
2.  It will specifically search for high-protein foods to meet this target.
3.  After the protein target is met, it will fill the remaining calorie budget with other foods to meet the carb and fat goals.
4.  We will also add logic to categorize foods into "meal types" to generate more logical meal combinations.

In [90]:
# @title Step 6: The Improved Generator Function (v2 - Protein Focused)
def generate_diet_plan_v2(df, daily_targets):
    """
    Generates a daily meal plan using a more intelligent, protein-focused greedy algorithm.

    Args:
        df (pd.DataFrame): The cleaned food dataframe.
        daily_targets (dict): A dictionary with target 'calories', 'protein', 'carbs', 'fat'.

    Returns:
        dict: A structured meal plan.
    """
    meal_plan = {
        'Breakfast': {'items': [], 'totals': {'Calories': 0, 'Protein': 0, 'Fat': 0, 'Carbs': 0}},
        'Lunch': {'items': [], 'totals': {'Calories': 0, 'Protein': 0, 'Fat': 0, 'Carbs': 0}},
        'Dinner': {'items': [], 'totals': {'Calories': 0, 'Protein': 0, 'Fat': 0, 'Carbs': 0}},
    }
    meal_distribution = {'Breakfast': 0.25, 'Lunch': 0.40, 'Dinner': 0.35}

    # Define some food categories for better meal cohesion
    # We can infer these from the 'Category' column
    breakfast_foods = df[df['Category'].str.contains('Breakfast', case=False, na=False)]
    main_course_foods = df[df['Category'].str.contains('Entrees|Prepared Meals|Meat', case=False, na=False)]

    for meal_name, percentage in meal_distribution.items():
        meal_targets = {k: v * percentage for k, v in daily_targets.items()}
        meal_df = meal_plan[meal_name]

        # --- PROTEIN FIRST ---
        protein_needed = meal_targets['protein']
        # Find high-protein foods (protein > 10g per 100g)
        high_protein_foods = df[df['Protein'] > 10]

        # Use specific meal-type foods if available
        if meal_name == 'Breakfast' and not breakfast_foods.empty:
            search_df = breakfast_foods[breakfast_foods['Protein'] > 5] # Lower protein threshold for breakfast
        elif not main_course_foods.empty:
            search_df = main_course_foods[main_course_foods['Protein'] > 10]
        else: # Fallback
            search_df = high_protein_foods

        if not search_df.empty:
            food_sample = search_df.sample(1).iloc[0]
            # Calculate portion to meet ~75% of meal's protein target
            target_protein_for_item = protein_needed * 0.75
            portion_g = round((target_protein_for_item / food_sample['Protein']) * 100) if food_sample['Protein'] > 0 else 0
            portion_g = min(max(portion_g, 50), 350)

            # Add this high-protein item
            portion_multiplier = portion_g / 100.0
            item = {
                'Food': food_sample['Food'], 'Portion (g)': portion_g,
                'Calories': round(food_sample['Calories'] * portion_multiplier),
                'Protein': round(food_sample['Protein'] * portion_multiplier),
                'Fat': round(food_sample['Fat'] * portion_multiplier),
                'Carbs': round(food_sample['Carbs'] * portion_multiplier)
            }
            meal_df['items'].append(item)
            for key in ['Calories', 'Protein', 'Fat', 'Carbs']: meal_df['totals'][key] += item[key]


        # --- FILLER FOODS FOR CALORIES ---
        calories_needed = meal_targets['calories'] - meal_df['totals']['Calories']
        while calories_needed > 100 and len(meal_df['items']) < 3:
            # Find foods to fill the rest, prioritizing carbs
            filler_foods = df[df['Carbs'] > 15]
            if filler_foods.empty: break # stop if no filler foods found
            food_sample = filler_foods.sample(1).iloc[0]

            # Calculate portion to fill remaining calories
            portion_g = round((calories_needed / food_sample['Calories']) * 100) if food_sample['Calories'] > 0 else 0
            portion_g = min(max(portion_g, 50), 300)

            portion_multiplier = portion_g / 100.0
            item = {
                'Food': food_sample['Food'], 'Portion (g)': portion_g,
                'Calories': round(food_sample['Calories'] * portion_multiplier),
                'Protein': round(food_sample['Protein'] * portion_multiplier),
                'Fat': round(food_sample['Fat'] * portion_multiplier),
                'Carbs': round(food_sample['Carbs'] * portion_multiplier)
            }
            meal_df['items'].append(item)
            for key in ['Calories', 'Protein', 'Fat', 'Carbs']: meal_df['totals'][key] += item[key]
            calories_needed = meal_targets['calories'] - meal_df['totals']['Calories']

    return meal_plan

# --- Let's test the v2 generator with the same user profile ---
print("--- Testing Improved Generator (v2) ---")
final_diet_plan_v2 = generate_diet_plan_v2(df_food_cleaned, user_macros)

print("\n\n" + "="*20 + " GENERATED DIET PLAN V2 " + "="*20)
print(json.dumps(final_diet_plan_v2, indent=2))

# --- Review and Compare Totals for V2 ---
total_calories_v2 = sum(final_diet_plan_v2[m]['totals']['Calories'] for m in final_diet_plan_v2)
total_protein_v2 = sum(final_diet_plan_v2[m]['totals']['Protein'] for m in final_diet_plan_v2)
total_carbs_v2 = sum(final_diet_plan_v2[m]['totals']['Carbs'] for m in final_diet_plan_v2)
total_fat_v2 = sum(final_diet_plan_v2[m]['totals']['Fat'] for m in final_diet_plan_v2)

print("\n\n" + "="*20 + " V2 PLAN TOTALS vs. TARGETS " + "="*20)
print(f"Calories: {total_calories_v2} / {user_macros['calories']} kcal")
print(f"Protein:  {total_protein_v2}g / {user_macros['protein']}g")
print(f"Carbs:    {total_carbs_v2}g / {user_macros['carbs']}g")
print(f"Fat:      {total_fat_v2}g / {user_macros['fat']}g")

--- Testing Improved Generator (v2) ---


==================== GENERATED DIET PLAN V2 ====================
{
  "Breakfast": {
    "items": [
      {
        "Food": "ORIGINAL FULLY COOKED PORK SAUSAGE CRUMBLES, ORIGINAL",
        "Portion (g)": 171,
        "Calories": 734,
        "Protein": 21,
        "Fat": 67,
        "Carbs": 3
      }
    ],
    "totals": {
      "Calories": 734,
      "Protein": 21,
      "Fat": 67,
      "Carbs": 3
    }
  },
  "Lunch": {
    "items": [
      {
        "Food": "ORGANIC GRASS FED BEEF",
        "Portion (g)": 191,
        "Calories": 392,
        "Protein": 34,
        "Fat": 27,
        "Carbs": 3
      },
      {
        "Food": "LEMON HERB SHRIMP WITH BARLEY, SPINACH, SWEET POTATO & CHICKPEAS",
        "Portion (g)": 241,
        "Calories": 214,
        "Protein": 9,
        "Fat": 4,
        "Carbs": 36
      }
    ],
    "totals": {
      "Calories": 606,
      "Protein": 43,
      "Fat": 31,
      "Carbs": 39
    }
  },
  "Dinner": {
   

### Step 7: Final Refinement (V3 - Balanced Algorithm)

**Analysis:** Our `v2` algorithm successfully prioritized protein but drastically overshot the calorie and fat targets. This happened because our high-protein food choices were also very high in fat.

**Improvement (V3):** We will create a more balanced algorithm that searches for **lean protein sources**.
1.  We will calculate a **"leanness score"** for each food (Protein grams / Calorie count). A higher score means more protein per calorie.
2.  When selecting the primary protein source for a meal, we will filter for foods with a **good leanness score**.
3.  We will also add a hard cap to the portion size calculation to prevent a single item from exceeding the meal's entire calorie budget. This acts as a crucial safety check.

In [91]:
# @title Step 7: The Final Generator Function (v3 - Balanced)

# First, let's create a 'Leanness' score in our dataframe
# Leanness = Protein / Calories. We'll ignore division-by-zero warnings for now.
with np.errstate(divide='ignore', invalid='ignore'):
    df_food_cleaned['Leanness'] = (df_food_cleaned['Protein'] * 4) / df_food_cleaned['Calories']
# Replace infinite values with 0 and fill NaNs
df_food_cleaned.replace([np.inf, -np.inf], 0, inplace=True)
df_food_cleaned['Leanness'].fillna(0, inplace=True)


def generate_diet_plan_v3(df, daily_targets):
    """
    Generates a balanced meal plan by prioritizing LEAN protein sources.
    """
    meal_plan = {
        'Breakfast': {'items': [], 'totals': {'Calories': 0, 'Protein': 0, 'Fat': 0, 'Carbs': 0}},
        'Lunch': {'items': [], 'totals': {'Calories': 0, 'Protein': 0, 'Fat': 0, 'Carbs': 0}},
        'Dinner': {'items': [], 'totals': {'Calories': 0, 'Protein': 0, 'Fat': 0, 'Carbs': 0}},
    }
    meal_distribution = {'Breakfast': 0.25, 'Lunch': 0.40, 'Dinner': 0.35}

    for meal_name, percentage in meal_distribution.items():
        meal_targets = {k: v * percentage for k, v in daily_targets.items()}
        meal_df = meal_plan[meal_name]

        # --- LEAN PROTEIN FIRST ---
        protein_needed = meal_targets['protein']
        # Find LEAN, high-protein foods. Leanness > 0.3 means at least 30% of calories from protein.
        lean_protein_foods = df[(df['Protein'] > 10) & (df['Leanness'] > 0.3)]

        if not lean_protein_foods.empty:
            # Add one lean protein source
            food_sample = lean_protein_foods.sample(1).iloc[0]

            # --- CRITICAL SAFETY CHECK ---
            # Calculate portion for protein, BUT cap it by the meal's calorie budget.
            # Portion for Protein:
            target_protein_for_item = protein_needed * 0.8 # Target 80% of protein from this item
            p_portion_g = round((target_protein_for_item / food_sample['Protein']) * 100) if food_sample['Protein'] > 0 else 100
            # Portion for Calories:
            c_portion_g = round((meal_targets['calories'] / food_sample['Calories']) * 100) if food_sample['Calories'] > 0 else 100
            # Choose the SMALLER of the two portions to avoid blowing a budget.
            portion_g = min(p_portion_g, c_portion_g)
            portion_g = min(max(portion_g, 50), 300) # Clamp to realistic serving

            # Add the item
            portion_multiplier = portion_g / 100.0
            item = {
                'Food': food_sample['Food'], 'Portion (g)': portion_g,
                'Calories': round(food_sample['Calories'] * portion_multiplier),
                'Protein': round(food_sample['Protein'] * portion_multiplier),
                'Fat': round(food_sample['Fat'] * portion_multiplier),
                'Carbs': round(food_sample['Carbs'] * portion_multiplier)
            }
            meal_df['items'].append(item)
            for key in ['Calories', 'Protein', 'Fat', 'Carbs']: meal_df['totals'][key] += item[key]

        # --- FILLER FOODS (UNCHANGED) ---
        calories_needed = meal_targets['calories'] - meal_df['totals']['Calories']
        while calories_needed > 100 and len(meal_df['items']) < 3:
            filler_foods = df[(df['Carbs'] > 15) & (df['Leanness'] < 0.2)] # Find carb sources that aren't also high-protein
            if filler_foods.empty: break
            food_sample = filler_foods.sample(1).iloc[0]

            portion_g = round((calories_needed / food_sample['Calories']) * 100) if food_sample['Calories'] > 0 else 0
            portion_g = min(max(portion_g, 50), 250)

            portion_multiplier = portion_g / 100.0
            item = {
                'Food': food_sample['Food'], 'Portion (g)': portion_g,
                'Calories': round(food_sample['Calories'] * portion_multiplier),
                'Protein': round(food_sample['Protein'] * portion_multiplier),
                'Fat': round(food_sample['Fat'] * portion_multiplier),
                'Carbs': round(food_sample['Carbs'] * portion_multiplier)
            }
            meal_df['items'].append(item)
            for key in ['Calories', 'Protein', 'Fat', 'Carbs']: meal_df['totals'][key] += item[key]
            calories_needed = meal_targets['calories'] - meal_df['totals']['Calories']

    return meal_plan

# --- Let's test the v3 generator ---
print("--- Testing Final Balanced Generator (v3) ---")
final_diet_plan_v3 = generate_diet_plan_v3(df_food_cleaned, user_macros)

print("\n\n" + "="*20 + " GENERATED DIET PLAN V3 " + "="*20)
print(json.dumps(final_diet_plan_v3, indent=2))

# --- Review and Compare Totals for V3 ---
total_calories_v3 = sum(final_diet_plan_v3[m]['totals']['Calories'] for m in final_diet_plan_v3)
total_protein_v3 = sum(final_diet_plan_v3[m]['totals']['Protein'] for m in final_diet_plan_v3)
total_carbs_v3 = sum(final_diet_plan_v3[m]['totals']['Carbs'] for m in final_diet_plan_v3)
total_fat_v3 = sum(final_diet_plan_v3[m]['totals']['Fat'] for m in final_diet_plan_v3)

print("\n\n" + "="*20 + " V3 PLAN TOTALS vs. TARGETS " + "="*20)
print(f"Calories: {total_calories_v3} / {user_macros['calories']} kcal")
print(f"Protein:  {total_protein_v3}g / {user_macros['protein']}g")
print(f"Carbs:    {total_carbs_v3}g / {user_macros['carbs']}g")
print(f"Fat:      {total_fat_v3}g / {user_macros['fat']}g")

C:\Users\MUHAMMAD ZAIN\AppData\Local\Temp\ipykernel_14996\2223629216.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_food_cleaned['Leanness'].fillna(0, inplace=True)


--- Testing Final Balanced Generator (v3) ---


==================== GENERATED DIET PLAN V3 ====================
{
  "Breakfast": {
    "items": [
      {
        "Food": "26-28 OZ H&G FLOUNDER",
        "Portion (g)": 182,
        "Calories": 129,
        "Protein": 23,
        "Fat": 3,
        "Carbs": 0
      },
      {
        "Food": "JELLIED CRANBERRY SAUCE, JELLIED CRANBERRY",
        "Portion (g)": 159,
        "Calories": 250,
        "Protein": 0,
        "Fat": 0,
        "Carbs": 64
      }
    ],
    "totals": {
      "Calories": 379,
      "Protein": 23,
      "Fat": 3,
      "Carbs": 64
    }
  },
  "Lunch": {
    "items": [
      {
        "Food": "RICOTTA PART SKIM CHEESE, RICOTTA",
        "Portion (g)": 231,
        "Calories": 365,
        "Protein": 36,
        "Fat": 20,
        "Carbs": 12
      },
      {
        "Food": "CHIPOTLE HUMMUS",
        "Portion (g)": 113,
        "Calories": 242,
        "Protein": 8,
        "Fat": 14,
        "Carbs": 32
      }
 